# Task notebook
The tasks can be completed in the notebook.

Included:
- `src/models.py`
- `src/tasks.py` 
- `tests/test_task.py`

If you prefer to work outside of the notebook.

## Goal

Build a small subset of a RAG backend:

- ingestion / chunking
- retrieval / ranking
- answer payload construction

**Recommended time spend:** 1–3 hours

### Dependencies 
if you are not working in notebook

Install test dependency:

```bash
python -m pip install pytest
```

In [ ]:
from dataclasses import dataclass, field
from typing import Any
import math
import re
from collections import defaultdict

## Shared models and sample data

The cells below define a simplified version of models and a small dataset.

You should use these models in your implementation.

In [ ]:

@dataclass(slots=True)
class DocumentSection:
    section_type: str
    content: str
    heading: str | None = None
    page_number: int | None = None
    metadata: dict[str, Any] = field(default_factory=dict)


@dataclass(slots=True)
class Document:
    document_id: str
    source_type: str
    source_ref: str
    title: str
    sections: list[DocumentSection] = field(default_factory=list)
    metadata: dict[str, Any] = field(default_factory=dict)


@dataclass(slots=True)
class Chunk:
    chunk_id: str
    document_id: str
    text: str
    section_type: str
    page_number: int | None = None
    metadata: dict[str, Any] = field(default_factory=dict)


@dataclass(slots=True)
class RetrievedChunk:
    chunk_id: str
    document_id: str
    text: str
    score: float
    section_type: str
    page_number: int | None = None
    metadata: dict[str, Any] = field(default_factory=dict)


@dataclass(slots=True)
class Query:
    text: str
    top_k: int = 5
    metadata: dict[str, Any] = field(default_factory=dict)

In [ ]:

sample_documents = [
    Document(
        document_id="KB-001",
        source_type="kb",
        source_ref="KB-001",
        title="Reset VPN Access",
        metadata={"product": "vpn", "audience": "staff"},
        sections=[
            DocumentSection("title", "Reset VPN Access"),
            DocumentSection("description", "User cannot connect after a password change."),
            DocumentSection("solution", "Remove the saved VPN profile, sign in again, and re-approve MFA."),
            DocumentSection("notes", "Applies to managed Windows laptops."),
        ],
    ),
    Document(
        document_id="KB-002",
        source_type="kb",
        source_ref="KB-002",
        title="Install Office 365",
        metadata={"product": "office", "audience": "staff"},
        sections=[
            DocumentSection("title", "Install Office 365"),
            DocumentSection("description", "Guide for installing Office applications."),
            DocumentSection("solution", "Open Company Portal, search for Microsoft 365 Apps, then click Install."),
            DocumentSection("notes", "Requires an active staff account."),
        ],
    ),
    Document(
        document_id="PDF-001",
        source_type="pdf",
        source_ref="vpn-manual.pdf",
        title="VPN Manual",
        metadata={"product": "vpn", "audience": "all"},
        sections=[
            DocumentSection("page_text", "Open the VPN client and select your university profile.", page_number=3),
            DocumentSection("page_text", "If login fails after a password reset, remove the old saved profile.", page_number=4),
            DocumentSection("page_text", "Approve the MFA request on your phone to complete sign-in.", page_number=4),
        ],
    ),
]

## Task 1 - Chunking with metadata preservation

### Your task

Implement `chunk_document(...)`.

### Requirements

1. Create one or more `Chunk` objects per document section.
2. Preserve important metadata:
   - `source_type`
   - `source_ref`
   - document-level metadata
   - section-level metadata
3. If a section's content is longer than `max_chars`, split it into multiple chunks.
4. Chunk IDs must be deterministic and easy to inspect, for example:
   - `KB-001:title:0`
   - `KB-001:solution:0`
   - `KB-001:solution:1`
5. Ignore sections whose content is empty or only whitespace.
6. Keep `page_number` when present.

### Suggested behavior

A simple fixed-size splitter is enough for this exercise.  
You do **not** need to implement semantic chunking.

### Deliverable

Write `chunk_document(...)` so the tests below pass.

In [ ]:
def split_text_fixed(text: str, max_chars: int) -> list[str]:
    '''
    Split text into fixed-size chunks.

    Expected behavior:
    - strip surrounding whitespace
    - return [] for empty text
    - if max_chars <= 0, raise ValueError
    '''
    if max_chars <= 0:
        raise ValueError("max_chars must be greater than 0")

    stripped = text.strip()
    if not stripped:
        return []

    return [stripped[i:i + max_chars] for i in range(0, len(stripped), max_chars)]


def chunk_document(document: Document, max_chars: int = 80) -> list[Chunk]:
    '''
    Convert a Document into retrieval Chunk objects.

    Rules:
    - preserve document and section metadata
    - skip empty sections
    - split long content using split_text_fixed
    - use deterministic chunk IDs
    '''
    chunks: list[Chunk] = []
    section_type_counter: dict[str, int] = defaultdict(int)

    for section in document.sections:
        parts = split_text_fixed(section.content, max_chars)
        if not parts:
            continue

        base_index = section_type_counter[section.section_type]

        metadata = {
            "source_type": document.source_type,
            "source_ref": document.source_ref,
            **document.metadata,
            **section.metadata,
        }

        for index, part in enumerate(parts):
            chunks.append(
                Chunk(
                    chunk_id=f"{document.document_id}:{section.section_type}:{base_index + index}",
                    document_id=document.document_id,
                    text=part,
                    section_type=section.section_type,
                    page_number=section.page_number,
                    metadata=metadata.copy(),
                )
            )

        section_type_counter[section.section_type] += len(parts)

    return chunks

In [ ]:

# Task 1 tests

chunks = chunk_document(sample_documents[0], max_chars=40)

assert chunks, "Expected at least one chunk"
assert all(isinstance(chunk, Chunk) for chunk in chunks)
assert all(chunk.document_id == "KB-001" for chunk in chunks)
assert all(chunk.metadata["source_type"] == "kb" for chunk in chunks)
assert all(chunk.metadata["source_ref"] == "KB-001" for chunk in chunks)
assert any(chunk.section_type == "solution" for chunk in chunks)

solution_chunks = [chunk for chunk in chunks if chunk.section_type == "solution"]
assert len(solution_chunks) >= 2, "Expected the solution section to be split into multiple chunks"

empty_doc = Document(
    document_id="KB-EMPTY",
    source_type="kb",
    source_ref="KB-EMPTY",
    title="Empty doc",
    sections=[
        DocumentSection("description", "   "),
        DocumentSection("solution", ""),
    ],
)

assert chunk_document(empty_doc, max_chars=20) == []

try:
    split_text_fixed("hello", 0)
    raise AssertionError("Expected ValueError when max_chars <= 0")
except ValueError:
    pass

print("Task 1 tests passed")

## Task 2 - Hybrid retrieval with metadata filtering

### Your task

Implement a simplified hybrid retriever that:

1. filters by metadata
2. fuses dense and lexical results using **Reciprocal Rank Fusion (RRF)**
3. boosts `solution` chunks slightly
4. returns the top `query.top_k` results

### Exact fusion formula

For each ranked list:

`rrf_score += 1 / (rank_constant + rank_position)`

Use:
- rank positions starting at **1**
- `rank_constant=60` by default

After fusion:
- if `section_type == "solution"`, multiply the fused score by `solution_boost`

### Metadata filtering rules

A chunk matches only if all provided metadata key/value pairs match exactly.

Example:

```python
query.metadata = {"product": "vpn"}
```

should keep only chunks where:

```python
chunk.metadata["product"] == "vpn"
```

### Deliverable

Implement:
- `filter_chunks_by_metadata`
- `hybrid_retrieve`

In [ ]:
def filter_chunks_by_metadata(
    chunks: list[RetrievedChunk],
    metadata_filter: dict[str, Any] | None,
) -> list[RetrievedChunk]:
    '''
    Return only chunks that match every key/value pair in metadata_filter.
    If metadata_filter is empty or None, return the input unchanged.
    '''
    if not metadata_filter:
        return chunks

    return [
        chunk
        for chunk in chunks
        if all(chunk.metadata.get(key) == value for key, value in metadata_filter.items())
    ]


def hybrid_retrieve(
    *,
    dense_results: list[RetrievedChunk],
    lexical_results: list[RetrievedChunk],
    query: Query,
    rank_constant: int = 60,
    solution_boost: float = 1.15,
) -> list[RetrievedChunk]:
    '''
    Fuse dense_results and lexical_results with Reciprocal Rank Fusion.

    Requirements:
    - apply query.metadata filtering before fusion
    - merge duplicates by chunk_id
    - keep document_id, text, section_type, page_number, metadata
    - assign the fused score to the returned RetrievedChunk.score
    - apply solution_boost after fusion
    - sort by score descending, then chunk_id ascending
    - return at most query.top_k items
    '''
    filtered_dense = filter_chunks_by_metadata(dense_results, query.metadata)
    filtered_lexical = filter_chunks_by_metadata(lexical_results, query.metadata)

    fused_scores: dict[str, float] = defaultdict(float)
    chunk_by_id: dict[str, RetrievedChunk] = {}

    for ranked_list in (filtered_dense, filtered_lexical):
        for rank_position, chunk in enumerate(ranked_list, start=1):
            fused_scores[chunk.chunk_id] += 1 / (rank_constant + rank_position)
            chunk_by_id.setdefault(chunk.chunk_id, chunk)

    fused_results: list[RetrievedChunk] = []
    for chunk_id, score in fused_scores.items():
        original = chunk_by_id[chunk_id]
        final_score = score
        if original.section_type == "solution":
            final_score *= solution_boost

        fused_results.append(
            RetrievedChunk(
                chunk_id=original.chunk_id,
                document_id=original.document_id,
                text=original.text,
                score=final_score,
                section_type=original.section_type,
                page_number=original.page_number,
                metadata=original.metadata.copy(),
            )
        )

    fused_results.sort(key=lambda chunk: (-chunk.score, chunk.chunk_id))
    return fused_results[:query.top_k]

In [ ]:

# Task 2 sample retrieval inputs

dense_results = [
    RetrievedChunk(
        chunk_id="KB-001:solution:0",
        document_id="KB-001",
        text="Remove the saved VPN profile.",
        score=0.91,
        section_type="solution",
        metadata={"source_type": "kb", "source_ref": "KB-001", "product": "vpn"},
    ),
    RetrievedChunk(
        chunk_id="PDF-001:page_text:1",
        document_id="PDF-001",
        text="If login fails after a password reset, remove the old saved profile.",
        score=0.89,
        section_type="page_text",
        page_number=4,
        metadata={"source_type": "pdf", "source_ref": "vpn-manual.pdf", "product": "vpn"},
    ),
    RetrievedChunk(
        chunk_id="KB-002:solution:0",
        document_id="KB-002",
        text="Open Company Portal and install Microsoft 365 Apps.",
        score=0.51,
        section_type="solution",
        metadata={"source_type": "kb", "source_ref": "KB-002", "product": "office"},
    ),
]

lexical_results = [
    RetrievedChunk(
        chunk_id="PDF-001:page_text:1",
        document_id="PDF-001",
        text="If login fails after a password reset, remove the old saved profile.",
        score=9.2,
        section_type="page_text",
        page_number=4,
        metadata={"source_type": "pdf", "source_ref": "vpn-manual.pdf", "product": "vpn"},
    ),
    RetrievedChunk(
        chunk_id="KB-001:solution:0",
        document_id="KB-001",
        text="Remove the saved VPN profile.",
        score=8.8,
        section_type="solution",
        metadata={"source_type": "kb", "source_ref": "KB-001", "product": "vpn"},
    ),
    RetrievedChunk(
        chunk_id="KB-002:solution:0",
        document_id="KB-002",
        text="Open Company Portal and install Microsoft 365 Apps.",
        score=7.0,
        section_type="solution",
        metadata={"source_type": "kb", "source_ref": "KB-002", "product": "office"},
    ),
]

In [ ]:

# Task 2 tests

query = Query(text="How do I reset VPN after password change?", top_k=3, metadata={"product": "vpn"})
results = hybrid_retrieve(
    dense_results=dense_results,
    lexical_results=lexical_results,
    query=query,
    rank_constant=60,
    solution_boost=1.10,
)

assert len(results) == 2, "Only vpn chunks should remain after metadata filtering"
assert results[0].chunk_id == "KB-001:solution:0", "Solution chunk should rank first after boost"
assert results[1].chunk_id == "PDF-001:page_text:1"
assert results[0].score > results[1].score

no_filter_results = hybrid_retrieve(
    dense_results=dense_results,
    lexical_results=lexical_results,
    query=Query(text="install office", top_k=5),
)

assert len(no_filter_results) == 3, "Duplicate chunk_ids should be merged"
assert len({chunk.chunk_id for chunk in no_filter_results}) == 3

filtered = filter_chunks_by_metadata(dense_results, {"product": "vpn"})
assert len(filtered) == 2
assert all(chunk.metadata["product"] == "vpn" for chunk in filtered)

print("Task 2 tests passed")

## Task 3 - Build an answer payload with citations

### Your task

Implement `build_answer_payload(...)`.

For this exercise, there is **no LLM call**.  
Instead, simulate the answer using the top retrieved chunks.

### Requirements

Return a dictionary with this shape:

```python
{
    "answer": "...",
    "citations": [
        {
            "chunk_id": "...",
            "document_id": "...",
            "section_type": "...",
            "page_number": 4,
            "source_type": "pdf",
            "source_ref": "vpn-manual.pdf",
        }
    ],
    "metadata": {
        "query": "...",
        "result_count": 2,
    },
}
```

### Answer rules

1. If no results are provided, return:
   - `"answer": "I could not find relevant support content."`
   - empty citations
2. Otherwise:
   - build a short answer by concatenating up to `max_citation_chunks` retrieved chunk texts
   - preserve ranking order
3. Citations should:
   - be unique by `chunk_id`
   - follow the same order as the ranked results
   - include only the required fields shown above

### Deliverable

Implement `build_answer_payload(...)`.

In [ ]:
def build_answer_payload(
    query: Query,
    retrieved_chunks: list[RetrievedChunk],
    max_citation_chunks: int = 3,
) -> dict[str, Any]:
    '''
    Build a response payload with:
    - answer
    - citations
    - metadata

    Do not call any external model.
    '''
    if not retrieved_chunks:
        return {
            "answer": "I could not find relevant support content.",
            "citations": [],
            "metadata": {
                "query": query.text,
                "result_count": 0,
            },
        }

    answer_chunks = retrieved_chunks[:max_citation_chunks]
    answer = " ".join(chunk.text for chunk in answer_chunks)

    citations = []
    seen_chunk_ids: set[str] = set()
    for chunk in retrieved_chunks:
        if chunk.chunk_id in seen_chunk_ids:
            continue
        seen_chunk_ids.add(chunk.chunk_id)
        citations.append(
            {
                "chunk_id": chunk.chunk_id,
                "document_id": chunk.document_id,
                "section_type": chunk.section_type,
                "page_number": chunk.page_number,
                "source_type": chunk.metadata["source_type"],
                "source_ref": chunk.metadata["source_ref"],
            }
        )
        if len(citations) >= max_citation_chunks:
            break

    return {
        "answer": answer,
        "citations": citations,
        "metadata": {
            "query": query.text,
            "result_count": len(retrieved_chunks),
        },
    }

In [ ]:

# Task 3 tests

ranked_results = hybrid_retrieve(
    dense_results=dense_results,
    lexical_results=lexical_results,
    query=Query(text="vpn reset", top_k=5, metadata={"product": "vpn"}),
)

payload = build_answer_payload(Query(text="vpn reset", top_k=5), ranked_results, max_citation_chunks=2)

assert isinstance(payload, dict)
assert set(payload.keys()) == {"answer", "citations", "metadata"}
assert payload["metadata"]["result_count"] == len(ranked_results)
assert payload["metadata"]["query"] == "vpn reset"
assert len(payload["citations"]) == 2
assert payload["citations"][0]["chunk_id"] == ranked_results[0].chunk_id
assert "source_type" in payload["citations"][0]
assert "source_ref" in payload["citations"][0]
assert payload["answer"], "Expected a non-empty answer"

empty_payload = build_answer_payload(Query(text="unknown"), [])
assert empty_payload["answer"] == "I could not find relevant support content."
assert empty_payload["citations"] == []

print("Task 3 tests passed")

## 
